# Loading of sessions

## Imports and session loading

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# point Python at your repo root (edit if needed)
from pathlib import Path
import sys, pickle

project_root = Path.cwd()
if not (project_root / "fiberphotometry").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

# imports from the lightweight package
from fiberphotometry.turnlab.context import set_context
from fiberphotometry.turnlab.linking import add_session_idx
from fiberphotometry.turnlab.config import SIGNAL_COLS
from fiberphotometry.turnlab.detect import make_turn_syllable_series
from fiberphotometry.turnlab.peth_plot import compute_turn_peths_from_merged
from fiberphotometry.turnlab.experiments import exp, run_turn_experiment, run_turn_experiments
from fiberphotometry.turnlab.cache_index import build_or_load_turn_index, slice_by_percentile, onset_mask_from_slice

# ---- load your saved state (sessions + merged_all) ----
load_filename = 'big_run_frac_1_ld10_jn24_with_merged.pkl'
print(f"Loading: {load_filename}")
with open(load_filename, 'rb') as f:
    state = pickle.load(f)

sessions        = state['sessions']
merged_all      = state['merged_all']
PROJECT_DIR_FULL= state.get('project_dir_full')

print(f"sessions: {len(sessions)} | merged_all: {merged_all.shape}")
set_context(sessions, PROJECT_DIR_FULL)

## Link sessions + prune columns + downcast dtypes

In [ ]:
import pandas as pd

merged_all = add_session_idx(merged_all, sessions)  # adds 'session_idx'

_need_detect = {'session_idx','frame_index','group','angular_velocity','velocity_px_s'}
_need_peth   = set(SIGNAL_COLS)
_keep = list((_need_detect | _need_peth) & set(merged_all.columns))

before_mb = merged_all.memory_usage(deep=True).sum()/(1024**2)
merged_all = merged_all.loc[:, _keep].copy()

# downcast
merged_all['session_idx']     = pd.to_numeric(merged_all['session_idx'], downcast='integer')
merged_all['frame_index']     = pd.to_numeric(merged_all['frame_index'], downcast='integer')
merged_all['group']           = merged_all['group'].astype('category')
merged_all['angular_velocity']= pd.to_numeric(merged_all['angular_velocity'], errors='coerce').astype('float32')
merged_all['velocity_px_s']   = pd.to_numeric(merged_all['velocity_px_s'], errors='coerce').astype('float32')
for c in SIGNAL_COLS:
    merged_all[c] = pd.to_numeric(merged_all[c], errors='coerce').astype('float32')

after_mb = merged_all.memory_usage(deep=True).sum()/(1024**2)
print(f"RAM: {before_mb:.1f}MB → {after_mb:.1f}MB | kept: {list(merged_all.columns)}")

## Quick baseline: detect + plot (relative, collapsed-by-bin)

In [ ]:
# detect ω-based turns with one bin (90) just to sanity-check pipeline
turn_syll, turn_on = make_turn_syllable_series(
    merged_all,
    session_idx_col='session_idx',
    frame_col='frame_index',
    omega_col='angular_velocity',
    speed_col='velocity_px_s',
    method='omega',
    granularity_percentiles=[90],
    fps=80,
    turn_start_pct=90,
    stop_ratio=0.6,
    min_bout_ms=100,
    merge_gap_ms=120,
    smooth_win=5,
    clip_hi_pct=99.5,
    speed_floor=2.0
)
df0 = merged_all.copy()
df0['turn_syllable'] = turn_syll
df0['turn_onset']    = turn_on

# plot: contra/ipsi, collapsed-by-bin, shared y per region
res0 = compute_turn_peths_from_merged(
    df0,
    desired_labels={'left_90','right_90'},
    label_col='turn_syllable',
    onset_col='turn_onset',
    lateralization='relative',
    collapse_relative='by_bin',
    link_y_by_region=True,
    combine_signals=True,
    out_dir=Path(PROJECT_DIR_FULL or '.')/'peth_quickcheck',
    save=True,
    min_traces_per_syllable=10
)

## Run a sweep of experiments (still from notebook)

https://chatgpt.com/c/68b9a33e-90f4-8320-ac0e-f433456e8029?model=gpt-5

## (Optional) Build a cache once, then slice percentile bins instantly

## Quick sanity helpers (optional cell)